In [1]:
import json
from shapely.geometry import Point, shape
import folium
import requests

obs_fp = 'data/observations.json'
hr_fp = 'data/historical_records.json'
wam_fp = 'src/WebAnnotationModel-2500-sample-results.json'

def get_obs_from_hr_uuid(hr_uuid, obs_fp=obs_fp):
    with open(obs_fp, 'r') as f:
        data = json.load(f)
    found_vals = []
    for obs in data['rde_objects']:
        if obs['documented_in'] == hr_uuid:
            found_vals.append(obs)
    return found_vals

def get_wam_annotation_from_record_id(rec_id, wam_fp=wam_fp):
    with open(wam_fp, 'r') as f:
        data = json.load(f)
    # return data
    for annotation in data:
        for b in annotation['body']:
            if 'record_id:' in b['value'] and rec_id in b['value']:
                return annotation
    return 

def get_record_id_from_hr_uuuid(hr_uuid, hr_fp=hr_fp):
    with open(hr_fp, 'r') as f:
        data = json.load(f)
    for hr in data['rde_objects']:
        if hr['uuid'] == hr_uuid:
            return hr['annotated_content'].get('record_id', None)
    return None

def zoom_level_from_bounds(max_lat, min_lat, max_lon, min_lon):
    lat_diff = max_lat - min_lat
    lon_diff = max_lon - min_lon
    max_diff = max(lat_diff, lon_diff)
    if max_diff < 0.005:
        return 15
    elif max_diff < 0.02:
        return 13
    elif max_diff < 0.1:
        return 11
    elif max_diff < 0.5:
        return 9
    elif max_diff < 2:
        return 7
    elif max_diff < 10:
        return 5
    else:
        return 3

def display_image_from_wam_obj(wam_obj):
    target = wam_obj['target']['id']
    # using ipython to display it
    from IPython.display import Image, display
    display(Image(url=target))

def extract_all_partial_matches_from_wam(wam_data):
    tpe = 'Dataset'
    purpose = 'linking'
    vals = list(filter(lambda b: b.get('type') == tpe and b.get('purpose') == purpose, wam_data['body']))
    # return vals
    if len(vals) == 0:
        return []
    else:
        hits = []
        for entry in vals[0]['value']:
            if entry['type'] == 'Partial match':
                hits.append((entry['value'], entry['source']))
    return hits

def get_gps_from_city_country(city, country):
    """
    Use Nominatim to get GPS coordinates from city and country.
    Returns (latitude, longitude) tuple or None if not found.
    """
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "city": city,
        "country": country,
        "format": "json",
        "limit": 1
    }
    headers = {
        "User-Agent": "europeana-timemachine-geocoder/1.0"
    }
    response = requests.get(url, params=params, headers=headers)
    if response.status_code == 200:
        results = response.json()
        if results:
            lat = float(results[0]["lat"])
            lon = float(results[0]["lon"])
            return lat, lon
    return None


/Users/viaccoz/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
from folium import IFrame
with open(wam_fp, 'r') as f:
    wam_objs = json.load(f)
wam_objs

with open(obs_fp, 'r') as f:
    obs_data = json.load(f)
obs_data

obs_coords_hr = {}
for obs in obs_data['rde_objects']:
    obs_uuid = obs['uuid']
    hr_uuid = obs['documented_in']
    coord = tuple(obs['geometry']['coordinates'])
    obs_coords_hr[obs_uuid] = (hr_uuid, coord)

obs_coords_hr
obs_coords_rec_id = {}
for obs_uuid, (hr_uuid, coord) in obs_coords_hr.items():
    if hr_uuid == 'c9709bed-7a79-580a-acf3-f1fb3f521d1c':
        # that damn portugal one who ended up everywhere.
        continue
    rec_id = get_record_id_from_hr_uuuid(hr_uuid, hr_fp=hr_fp)
    obs_coords_rec_id[obs_uuid] = (rec_id, coord)  


In [3]:
from tqdm.notebook import tqdm
located_rec_id = {v[0] for v in obs_coords_rec_id.values() if v[0] is not None}
len(located_rec_id)
all_rec_id = {}
for wam in wam_objs:
    for b in wam['body']:
        if 'record_id:' in b['value']:
            rec_id = b['value'].split('record_id:')[-1].strip()
            if rec_id not in located_rec_id:
                all_rec_id[rec_id] = wam
len(all_rec_id)
all_rec_id
country_city_pairs = set()
greogrouped_recs = {}
for rec_id, wam in all_rec_id.items():
    country = None
    city = None
    for b in wam['body']:
        if b.get('type') == 'TextualBody' and b.get('purpose') == 'identifying':
            if 'final_country:' in b['value']:
                country = b['value'].split('country:')[-1].strip()
            if 'city:' in b['value']:
                city = b['value'].split('city:')[-1].strip()
    if country is not None and city is not None:
        key = (country, city)
        country_city_pairs.add(key)
        greogrouped_recs.setdefault(key, []).append((rec_id, wam))

len(country_city_pairs)
with open('country_city_coords.json', 'r') as f:
    country_city_coords = json.load(f)

for country, city in tqdm(country_city_pairs):
    str_code = f'{country}, {city}'
    if not str_code in country_city_coords:
        coords = get_gps_from_city_country(city, country)
        country_city_coords[str_code] = coords

with open('country_city_coords.json', 'w') as f:
    json.dump(country_city_coords, f, indent=4)

  0%|          | 0/824 [00:00<?, ?it/s]

In [4]:
import pandas as pd
with open('src/WebAnnotationModel-2500-sample-results.json', 'r') as f:
    cont = json.load(f)

def flatten(li: list) -> list:
    return [item for sublist in li for item in sublist]

def extract_value_from_textual_body_with_semantic_prefix(purpose, value, sample):
    bodies = sample['body']
    for b in bodies:
        if b.get('purpose', None) == purpose and b.get('type', None) == 'TextualBody' and b.get('value', '').startswith(f'{value}:'):
            return b.get('value', '').replace(f'{value}: ', '')
    return None 

def extract_value_from_type_with_purpose(tpe, purpose, sample, str_cont = None):
    bodies = sample['body']
    for b in bodies:
        if b.get('type', None) == tpe and b.get('purpose', None) == purpose and (str_cont is None or str_cont in b.get('value', '')):
            return b.get('value', '')
    return None

def extract_value_from_textual_body_with_purpose(purpose, sample):
    return extract_value_from_type_with_purpose("TextualBody", purpose, sample)

def extract_target_img(sample):
    if sample.get('target', {}).get('type', None) == 'Image':
        return sample.get('target', {}).get('id', None)
    return None

def extract_dataset_tavily_results_in_array(sample):
    bodies = sample['body']
    res = []
    for b in bodies:
        if b.get('type', None) == 'Dataset' and b.get('purpose', None) == "searching":
            for entry in b.get('value', []):
                if 'search query' in entry.get('type', ''):
                    res.append('Query: ' + entry.get('value', ''))
                elif 'search result' in entry.get('type', ''):
                    res.append(f"Result: {entry.get('value', '')} - {entry.get('source', '')}")
    return res

def extract_google_search_results(sample):
    bodies = sample['body']
    res = []
    for b in bodies:
        if b.get('type', None) == 'Dataset' and b.get('purpose', None) == "linking":
            for entry in b.get('value', []):
                if 'Partial match' in entry.get('type', ''):
                    res.append(f"Partial match: {entry.get('value', '')} - {entry.get('source', '')}")
                elif 'Full match' in entry.get('type', ''):
                    res.append(f"Full match: {entry.get('value', '')} - {entry.get('source', '')}")
    return res

def extract_google_coordinates(sample):
    bodies = sample['body']
    res = []
    for b in bodies:
        if b.get('type', None) == 'Dataset' and b.get('purpose', None) == "linking":
            for entry in b.get('value', []):
                if coords := entry.get('object_coordinates', None):
                    if isinstance(coords[0], list) and isinstance(coords[0][0], list):
                        res.append(flatten(coords))
                    else:
                        res.append(coords)
    return res if len(res) >= 1 else None

def extract_landmark_information(sample):
    bodies = sample['body']
    res = []
    for b in bodies:
        if b.get('type', None) == 'Dataset' and b.get('purpose', None) == "identifying":
            for entry in b.get('value', []):
                if 'Landmark' in entry.get('type', ''):
                    val_str = entry.get('value', '')
                    if wd_id := entry.get('wikidata_id', None):
                        val_str += f" (https://www.wikidata.org/wiki/{wd_id})"
                    res.append(val_str)
    return res

def extract_landmark_coordinates(sample):
    bodies = sample['body']
    res = []
    for b in bodies:
        if b.get('type', None) == 'Dataset' and b.get('purpose', None) == "identifying":
            for entry in b.get('value', []):
                if coords := entry.get('coordinates', None):
                    res.append(coords)
    return res if len(res) >= 1 else None

json_path_parse_dict = {
    "is_postcard": lambda s: extract_value_from_textual_body_with_semantic_prefix("classifying", "postcard", s),
    "back_postcard": lambda s: extract_value_from_textual_body_with_semantic_prefix("classifying", "postcard's back", s),
    "final_place": lambda s: extract_value_from_textual_body_with_semantic_prefix("identifying", "final_place", s),
    "final_country": lambda s: extract_value_from_textual_body_with_semantic_prefix("identifying", "final_country", s),
    "final_city": lambda s: extract_value_from_textual_body_with_semantic_prefix("identifying", "final_city", s),
    "filename": lambda s: extract_value_from_textual_body_with_semantic_prefix("identifying", "filename", s),
    "record_id": lambda s: extract_value_from_textual_body_with_semantic_prefix("identifying", "record_id", s),
    "description": lambda s: extract_value_from_textual_body_with_purpose("describing", s),
    "transcription": lambda s: extract_value_from_textual_body_with_purpose("transcribing", s),
    "date": lambda s: extract_value_from_textual_body_with_purpose("dating", s),
    "europeana_link": lambda s: extract_value_from_type_with_purpose("SpecificResource", "linking", s, "europeana.eu"),
    "rights_attribution": lambda s: extract_value_from_type_with_purpose("SpecificResource", "tagging", s),
    "reason": lambda s: extract_value_from_textual_body_with_purpose("commenting", s),
    "img_src": extract_target_img,
    "tavily": extract_dataset_tavily_results_in_array,
    "google_search": extract_google_search_results,
    "google_coordinates": extract_google_coordinates,
    "landmarks_identified": extract_landmark_information,
    "landmarks_coordinates": extract_landmark_coordinates
}

sampled_values = []
for s in cont:
    vals = {}
    for k, v in json_path_parse_dict.items():
        vals[k] = v(s)
    sampled_values.append(vals)

df = pd.DataFrame(sampled_values)


In [5]:


# leaflet map with an image displayed on the marker when clicked
all_coords = [v[1] for v in obs_coords_rec_id.values()]
max_lat = max([coord[1] for coord in all_coords])
max_lon = max([coord[0] for coord in all_coords])
min_lat = min([coord[1] for coord in all_coords])
min_lon = min([coord[0] for coord in all_coords])
center_view = [(max_lat + min_lat) / 2, (max_lon + min_lon) / 2]

m = folium.Map(location=center_view, zoom_start=zoom_level_from_bounds(max_lat, min_lat, max_lon, min_lon))

for obs, (rec_id, coord) in obs_coords_rec_id.items():
    wam_obj = get_wam_annotation_from_record_id(rec_id, wam_fp=wam_fp)
    all_partials_matches = extract_all_partial_matches_from_wam(wam_obj)
    if len(all_partials_matches) > 0:
        target = wam_obj['target']['id']
        europeana_url = extract_value_from_type_with_purpose("SpecificResource", "linking", wam_obj, "europeana.eu")
        html = '<a href={} target="blank_"><img src="{}" style="max-width: 500px; height: auto;"></a>'.format(europeana_url, target)
        # html += '<button onclick="window.copyToClipboard(\'{}\');">Save Image</button>'.format(rec_id)
        html += '<br><b>Partial Matches:</b><br>'
        for match_value, match_source in all_partials_matches:
            html += '<img src="{}" style="max-width: 500px; height: auto;"> <br> (Source: <a href="{}" target="_blank">{}</a>)<br>'.format(match_value, match_source, match_source)
        iframe = IFrame(html, width=500, height=500)
        geo_j = folium.Marker(location=[coord[1], coord[0]])
        folium.Popup(iframe).add_to(geo_j)
        geo_j.add_to(m)


In [6]:
from io import StringIO
from folium.elements import *


for (country, city), rec_list in greogrouped_recs.items():
    str_code = f'{country}, {city}'
    coords = country_city_coords.get(str_code)
    # More robust check for valid coordinates
    if coords is None or not isinstance(coords, (list, tuple)) or len(coords) < 2:
        continue
    lat, lon = coords[0], coords[1]
    if lat is None or lon is None:
        continue
    try:
        lat, lon = float(lat), float(lon)
    except (ValueError, TypeError):
        continue
    
    html = ''
    total_amount = 0
    for rec_id, wam_inst in rec_list:
        all_partials_matches = extract_all_partial_matches_from_wam(wam_inst)
        if len(all_partials_matches) > 0:
            europeana_url = extract_value_from_type_with_purpose("SpecificResource", "linking", wam_inst, "europeana.eu")
            total_amount += 1
            target = wam_inst['target']['id']
            html += '<h3>Image #{}</h3>'.format(total_amount)
            html += '<a href="{}" target="_blank"><img src="{}" style="max-width: 500px; height: auto;"></a>'.format(europeana_url, target)
            # html += '<button onclick="copyToClipboard(\'{}\');">Copy Record ID</button>'.format(rec_id)
            html += '<br><b>Partial Matches:</b><br>'
            for match_value, match_source in all_partials_matches:
                html += '<img src="{}" style="max-width: 500px; height: auto;"> <br> (Source: <a href="{}" target="_blank">{}</a>)<br>'.format(match_value, match_source, match_source)
            html += '<br><hr><br>'
    if total_amount == 0:
        continue
    try:
        folium.CircleMarker(
            location=[lat, lon],
            radius=total_amount*3,
            color='purple',
            fill=True,
            fill_opacity=0.6,
            opacity=1,
            popup=folium.Popup(IFrame(f'<h2> {total_amount} Images for {str_code}'+html, width=500, height=500)),
            tooltip="total images: {}".format(total_amount)
        ).add_to(m)
    except ValueError as e:
        print(f"Skipping invalid coordinates for {str_code}: ({lat}, {lon}) - Error: {e}")
        continue

def generate_json_dict_from_df(df) -> str:
    str_buff = StringIO()
    str_buff.write('{\n')
    for index, row in df.iterrows():
        rec_id = row['record_id']
        str_buff.write(f'  "{rec_id}": "')
        str_buff.write(row.to_csv(header=False).strip().replace('\n', ' ').replace('\"', '\\"'))
        str_buff.write('",\n')
    str_buff.write('}\n')
    json_dict_str = str_buff.getvalue()
    str_buff.close()
    del str_buff
    return json_dict_str

str_buff = StringIO()
df.to_json(str_buff, orient='records', lines=True)

# JavaScript function to copy to clipboard
copy_to_clipboard_js = '''
console.log("Copy to clipboard script loaded.");
function copyToClipboard(text) {
    if (navigator.clipboard && navigator.clipboard.writeText) {
        navigator.clipboard.writeText(text).then(function() {
            alert('Record ID copied to clipboard: ' + text);
        }).catch(function(err) {
            console.error('Failed to copy:', err);
            alert('Failed to copy to clipboard');
        });
    } else {
        // Fallback for older browsers
        var textArea = document.createElement("textarea");
        textArea.value = text;
        document.body.appendChild(textArea);
        textArea.select();
        try {
            document.execCommand('copy');
            alert('Record ID copied to clipboard: ' + text);
        } catch (err) {
            console.error('Failed to copy:', err);
            alert('Failed to copy to clipboard');
        }
        document.body.removeChild(textArea);
    }
}
'''

m.get_root().script.add_child(Element(copy_to_clipboard_js))

m.save('europeana_partial_postcards.html')

In [7]:
wrong_monuments_ids = [
    "3806868c-462e-5f25-8487-682384c7fa35",
    "5bb5e324-252c-5ab9-982e-7321b3ea1065",
    "53e76637-ae14-557b-9775-bd6fb9a64fb3"
]